# 📈 Retorno Acumulado de Ativos

Digite o **ativo**, a **data de início** e a **data de fim** — o notebook devolve o retorno acumulado do período.

**Como usar no celular:**
1. Toque em ▶️ na célula **1️⃣** (leva ~30 segundos, é só na primeira vez que abrir).
2. Preencha os campos da célula **2️⃣** e toque em ▶️.
3. Para consultar outro ativo, é só mudar os campos e tocar em ▶️ de novo.

Se ficar parado por muito tempo, o Colab desconecta — nesse caso rode a célula 1️⃣ novamente.

**Aceita:** ações BR (`PETR4`), ETFs (`BOVA11`, `IVVB11`), FIIs (`HGLG11`), BDRs (`AAPL34`), ações EUA (`AAPL`, `NVDA`), ETFs EUA (`SPY`, `QQQ`), índices (`IBOV`, `SP500`, `NASDAQ`, `DOW`), câmbio (`DOLAR`), commodities (`OURO`) e cripto (`BTC`).

**Datas:** `dd/mm/aaaa` — ex: `05/01/2021`.

Fonte dos dados: Yahoo Finance.

In [ ]:
#@title 1️⃣ Preparar (rode uma vez, ao abrir o notebook) { display-mode: "form" }
# Instala a biblioteca de dados e carrega as funções.
# Fonte dos dados: Yahoo Finance.

!pip install -q yfinance

import re, warnings, logging, contextlib, io
from datetime import datetime, timedelta
import yfinance as yf

warnings.filterwarnings("ignore")
logging.getLogger("yfinance").setLevel(logging.CRITICAL)

ALIASES = {
    "IBOV": "^BVSP", "IBOVESPA": "^BVSP", "BVSP": "^BVSP", "IFIX": "IFIX.SA",
    "SP500": "^GSPC", "S&P500": "^GSPC", "SPX": "^GSPC", "GSPC": "^GSPC",
    "NASDAQ": "^IXIC", "IXIC": "^IXIC", "NDX": "^NDX", "NASDAQ100": "^NDX",
    "DOW": "^DJI", "DOWJONES": "^DJI", "DJI": "^DJI",
    "RUSSELL2000": "^RUT", "RUT": "^RUT", "VIX": "^VIX",
    "FTSE": "^FTSE", "DAX": "^GDAXI", "NIKKEI": "^N225",
    "DOLAR": "BRL=X", "USDBRL": "BRL=X", "EURBRL": "EURBRL=X", "EURUSD": "EURUSD=X",
    "OURO": "GC=F", "GOLD": "GC=F", "PETROLEO": "CL=F", "OIL": "CL=F",
    "BITCOIN": "BTC-USD", "BTC": "BTC-USD", "ETHEREUM": "ETH-USD", "ETH": "ETH-USD",
}

CONFUSOES = {
    "IBOV": "IBOV (o índice) ou BOVA11 (o ETF que replica o Ibovespa)",
    "SP500": "SP500 (o índice) ou IVVB11 / SPY / VOO (ETFs que o replicam)",
    "NASDAQ": "NASDAQ (o índice) ou QQQ (ETF) ou NASD11 (ETF na B3)",
    "DOW": "DOW (o índice Dow Jones) ou DIA (ETF)",
    "BTC": "BTC (bitcoin em dólar) ou BITH11 (ETF de cripto na B3)",
}

PADRAO_B3 = re.compile(r"^[A-Z]{4}\d{1,2}$")
FORMATOS_DATA = ("%d/%m/%Y", "%d/%m/%y", "%Y-%m-%d", "%d-%m-%Y")
SIMBOLO_MOEDA = {"BRL": "R$", "USD": "US$", "EUR": "€", "GBP": "£", "JPY": "¥"}


@contextlib.contextmanager
def sem_ruido():
    lixo = io.StringIO()
    with contextlib.redirect_stderr(lixo), contextlib.redirect_stdout(lixo):
        yield


def normalizar(entrada):
    return entrada.strip().upper().replace(" ", "")


def candidatos(entrada):
    codigo, lista = normalizar(entrada), []
    def add(s):
        if s and s not in lista:
            lista.append(s)
    if not codigo:
        return []
    if codigo in ALIASES:
        add(ALIASES[codigo])
    if codigo.startswith("^") or "." in codigo or "=" in codigo or "-" in codigo:
        add(codigo)
    if PADRAO_B3.match(codigo):
        add(codigo + ".SA")
    add(codigo)
    add(codigo + ".SA")
    return lista


def sugerir(entrada):
    codigo = normalizar(entrada)
    for chave in (codigo, codigo.rstrip("0123456789")):
        if chave in CONFUSOES:
            return CONFUSOES[chave]
    return None


def ler_data(texto):
    for formato in FORMATOS_DATA:
        try:
            return datetime.strptime(texto.strip(), formato)
        except ValueError:
            continue
    return None


def buscar_historico(entrada, inicio, fim):
    fim_exclusivo = fim + timedelta(days=1)
    for simbolo in candidatos(entrada):
        try:
            with sem_ruido():
                papel = yf.Ticker(simbolo)
                df = papel.history(start=inicio.strftime("%Y-%m-%d"),
                                   end=fim_exclusivo.strftime("%Y-%m-%d"),
                                   auto_adjust=False, actions=False)
        except Exception:
            continue
        if df is None or df.empty or "Close" not in df.columns:
            continue
        df = df.dropna(subset=["Close"])
        if len(df) < 2:
            continue
        moeda, nome = "", simbolo
        with sem_ruido():
            try:
                moeda = papel.fast_info.get("currency") or ""
            except Exception:
                pass
            try:
                d = papel.info
                nome = d.get("longName") or d.get("shortName") or simbolo
            except Exception:
                pass
        return simbolo, df, moeda, nome
    return None, None, None, None


def calcular(df):
    coluna = "Adj Close" if "Adj Close" in df.columns else "Close"
    total, preco = df[coluna].dropna(), df["Close"].dropna()
    ret_total = float(total.iloc[-1]) / float(total.iloc[0]) - 1
    ret_preco = float(preco.iloc[-1]) / float(preco.iloc[0]) - 1
    d0, d1 = total.index[0].to_pydatetime(), total.index[-1].to_pydatetime()
    anos = max((d1 - d0).days, 1) / 365.25
    diaria = total.pct_change().dropna()
    return {
        "data_ini": d0, "data_fim": d1,
        "preco_ini": float(preco.iloc[0]), "preco_fim": float(preco.iloc[-1]),
        "retorno_total": ret_total, "retorno_preco": ret_preco,
        "tem_proventos": coluna == "Adj Close" and abs(ret_total - ret_preco) > 1e-6,
        "anualizado": (1 + ret_total) ** (1 / anos) - 1 if anos >= 0.08 else None,
        "drawdown": float((total / total.cummax() - 1).min()),
        "volatilidade": float(diaria.std() * (252 ** 0.5)) if len(diaria) > 1 else None,
        "pregoes": len(total),
        "maxima": float(preco.max()), "minima": float(preco.min()),
        "serie": total,
    }


def pct(v):
    return f"{v * 100:+,.2f}%".replace(",", "@").replace(".", ",").replace("@", ".")


def dinheiro(v, s):
    t = f"{v:,.2f}".replace(",", "@").replace(".", ",").replace("@", ".")
    return f"{s} {t}" if s else t


def consultar(ativo, data_inicio, data_fim, mostrar_grafico=True):
    inicio, fim = ler_data(data_inicio), ler_data(data_fim)
    if not inicio or not fim:
        print("  Data inválida. Use dd/mm/aaaa, ex: 05/01/2021.")
        return
    if inicio >= fim:
        print("  A data de início precisa ser anterior à data de fim.")
        return

    print(f"  Buscando {normalizar(ativo)}...")
    simbolo, df, moeda, nome = buscar_historico(ativo, inicio, fim)
    if df is None:
        print(f"\n  Não encontrei dados para '{normalizar(ativo)}' nesse período.")
        dica = sugerir(ativo)
        if dica:
            print(f"  Você quis dizer: {dica}?")
        print("  Exemplos: PETR4, BOVA11, IBOV, AAPL, SPY, SP500, HGLG11, BTC")
        return

    m = calcular(df)
    c = SIMBOLO_MOEDA.get(moeda, moeda)
    br = lambda d: d.strftime("%d/%m/%Y")

    print()
    print("=" * 58)
    print(f"  {nome}")
    print(f"  {simbolo}  ·  {br(m['data_ini'])} → {br(m['data_fim'])}  ·  {m['pregoes']} pregões")
    print("=" * 58)
    print(f"\n  Preço inicial ......... {dinheiro(m['preco_ini'], c)}")
    print(f"  Preço final ........... {dinheiro(m['preco_fim'], c)}\n")
    print(f"  RETORNO ACUMULADO ..... {pct(m['retorno_total'])}")
    if m["tem_proventos"]:
        print(f"     · só variação de preço ....... {pct(m['retorno_preco'])}")
        print(f"     · com proventos reinvestidos . {pct(m['retorno_total'])}")
    if m["anualizado"] is not None:
        print(f"  Retorno anualizado .... {pct(m['anualizado'])} a.a.")
    print(f"\n  {dinheiro(1000, c)} investidos virariam "
          f"{dinheiro(1000 * (1 + m['retorno_total']), c)}\n")
    print(f"  Máxima no período ..... {dinheiro(m['maxima'], c)}")
    print(f"  Mínima no período ..... {dinheiro(m['minima'], c)}")
    print(f"  Queda máxima .......... {pct(m['drawdown'])}")
    if m["volatilidade"] is not None:
        print(f"  Volatilidade anual .... {m['volatilidade'] * 100:.2f}%")
    print("\n  Fonte: Yahoo Finance")
    print("=" * 58)

    if mostrar_grafico:
        import matplotlib.pyplot as plt
        acumulado = (m["serie"] / m["serie"].iloc[0] - 1) * 100
        fig, ax = plt.subplots(figsize=(7, 3.2))
        cor = "#1f7a4d" if m["retorno_total"] >= 0 else "#b02418"
        ax.plot(acumulado.index, acumulado.values, color=cor, linewidth=1.6)
        ax.fill_between(acumulado.index, 0, acumulado.values, color=cor, alpha=0.10)
        ax.axhline(0, color="#999999", linewidth=0.8)
        ax.set_title(f"{simbolo} — retorno acumulado (%)", fontsize=11, loc="left")
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(axis="y", alpha=0.25)
        plt.tight_layout()
        plt.show()


print("✅ Pronto. Agora é só usar a célula 2 abaixo.")

In [ ]:
#@title 2️⃣ Consultar — preencha e toque em ▶️ { display-mode: "form" }

Ativo = "PETR4" #@param {type:"string"}
Data_de_inicio = "01/01/2020" #@param {type:"string"}
Data_de_fim = "31/12/2024" #@param {type:"string"}
Mostrar_grafico = True #@param {type:"boolean"}

consultar(Ativo, Data_de_inicio, Data_de_fim, Mostrar_grafico)